# Neurodesk Evidence-Informed Workflow Notebook

This notebook has been rewritten to follow the workflow structure in the proposal and thesis slides.

It is organized into five connected stages:

1. **Stage 1: Structural MRI feature inputs**
2. **Stage 2: Literature-informed evidence framework**
3. **Stage 3: Optional multimodal extension**
4. **Stage 4: Patient-to-evidence comparison**
5. **Stage 5: Structured interpretive reporting**

The core idea is to combine **structural MRI outputs** with **literature evidence**, so interpretation is not based only on papers and not based only on ROI tables.


## Workflow Alignment With Proposal And Slides

This notebook now mirrors the project logic in the proposal:

- **Aim 1**: organize structural MRI outputs from FreeSurfer / FastSurfer / recon-all-clinical
- **Aim 2**: build a structured evidence framework from literature
- **Aim 3**: optionally include DWI, perfusion, cognitive, behavioural, or biomarker data
- **Comparison stage**: compare subject or group features against structured evidence
- **Reporting stage**: produce a traceable summary with explicit caveats and uncertainty

This notebook is designed as an **evidence-informed interpretive support workflow**, not a fully automated diagnostic system.


## Data Sources

This workflow can combine four input layers:

1. **Literature evidence sources**
   - PubMed / PMC URLs
   - local PDF files
   - other text sources describing disease-related imaging patterns
2. **Structural MRI feature table**
   - vertical CSV exported from FreeSurfer / FastSurfer / related pipelines
3. **Normative / control reference table**
   - optional control or age / sex adjusted reference values
4. **Complementary tables**
   - optional DWI, perfusion, cognitive, behavioural, or biomarker data

Generated database artifacts such as `literature_database.sql`, `literature_database.json`, and `literature_database.xlsx` are workflow outputs and should be regenerated locally instead of uploaded.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display, Markdown

project_dir = Path.cwd()
sys.path.append(str(project_dir))

from neurodesk_literature_to_pgsql import (
    analyze_sources,
    build_evidence_framework,
    build_interpretive_report,
    compare_features_to_literature,
    compare_to_normative_reference,
    literature_diseases_for_features,
    literature_features_for_disease,
    normalize_structural_dataframe,
    query_literature,
    query_structural_features,
    summarize_diagnosis_support,
    summarize_structural_features,
)


## Stage 1. Configure Inputs

### Structural MRI as the core input layer

This stage corresponds to **Aim 1** in the proposal.

Configure:

- literature sources for the evidence framework
- structural MRI feature CSV produced by FreeSurfer / FastSurfer / related pipelines
- optional normative reference table
- optional complementary tables
- output directory for generated literature database artifacts


In [2]:
# LITERATURE SOURCES FOR THE EVIDENCE FRAMEWORK
INPUTS = [
    # 'https://pubmed.ncbi.nlm.nih.gov/21810890/',
    # 'https://pubmed.ncbi.nlm.nih.gov/21325651/',
    # '/home/jovyan/Desktop/paper.pdf',
]

# OUTPUT DIRECTORY FOR LITERATURE DATABASE ARTIFACTS
OUTPUT_DIR = Path('./literature_outputs_workflow_demo')
LITERATURE_JSON_PATH = OUTPUT_DIR / 'literature_database.json'

# STRUCTURAL MRI FEATURE TABLE
# Actual FreeSurfer ROI feature table generated from OAS1_0003_MR1. Single-case prototype only; no group comparison.
STRUCTURAL_FEATURES_CSV_PATH = Path('/home/jovyan/oasis_selected/ipd_vertical_dataset/oasis_freesurfer_roi_vertical_OAS1_0003_single_case.csv')

# OPTIONAL NORMATIVE / CONTROL REFERENCE TABLE
NORMATIVE_CSV_PATH = Path('/home/jovyan/oasis_selected/ipd_vertical_dataset/oasis_freesurfer_roi_vertical_OAS1_0003_single_case.csv')
USE_NORMATIVE_REFERENCE = False

# OPTIONAL COMPLEMENTARY TABLES FOR AIM 3
DWI_CSV_PATH = None
PERFUSION_CSV_PATH = None
COGNITIVE_CSV_PATH = None
BIOMARKER_CSV_PATH = None

# AI BACKEND SETTINGS FOR LITERATURE EXTRACTION
AI_BACKEND = 'none'
AI_BASE_URL = 'http://127.0.0.1:11434/v1'
AI_MODEL = 'llama3.1:8b'
AI_API_KEY = None

settings_df = pd.DataFrame([
    {'setting': 'n_literature_inputs', 'value': len(INPUTS)},
    {'setting': 'output_dir', 'value': str(OUTPUT_DIR)},
    {'setting': 'literature_json_path', 'value': str(LITERATURE_JSON_PATH)},
    {'setting': 'structural_features_csv_path', 'value': str(STRUCTURAL_FEATURES_CSV_PATH)},
    {'setting': 'structural_features_exists', 'value': STRUCTURAL_FEATURES_CSV_PATH.exists()},
    {'setting': 'normative_csv_path', 'value': str(NORMATIVE_CSV_PATH)},
    {'setting': 'normative_csv_exists', 'value': NORMATIVE_CSV_PATH.exists()},
    {'setting': 'use_normative_reference', 'value': USE_NORMATIVE_REFERENCE},
    {'setting': 'ai_backend', 'value': AI_BACKEND},
])

display(settings_df)


metadata_warning_df = pd.DataFrame([
    {'issue': 'single_case_only', 'status': 'Only OAS1_0003_MR1 has measured FreeSurfer outputs; group comparison is not valid.'},
    {'issue': 'missing_demographics', 'status': 'sex is unknown and age/MMSE/CDR are missing unless participant metadata is re-uploaded.'},
    {'issue': 'evidence_status', 'status': 'Stage 4 matches are candidate evidence requiring manual review, not accepted evidence.'},
])
display(Markdown('### Data scope and limitations'))
display(metadata_warning_df)


,setting,value
0,n_literature_inputs,0
1,output_dir,literature_outputs_workflow_demo
2,literature_json_path,literature_outputs_workflow_demo/literature_da...
3,structural_features_csv_path,/home/jovyan/oasis_selected/ipd_vertical_datas...
4,structural_features_exists,True
5,normative_csv_path,/home/jovyan/oasis_selected/ipd_vertical_datas...
6,normative_csv_exists,True
7,use_normative_reference,False
8,ai_backend,none


### Data scope and limitations

,issue,status
0,single_case_only,Only OAS1_0003_MR1 has measured FreeSurfer out...
1,missing_demographics,sex is unknown and age/MMSE/CDR are missing un...
2,evidence_status,Stage 4 matches are candidate evidence requiri...


## Stage 1A. Load Structural MRI Features

### Expected feature table structure

This stage corresponds to the **structured feature outputs** described in the slides.

The feature table should ideally contain columns such as:

- `subject_id`
- `diagnosis`
- `sex`
- `roi_name`
- `imaging_metric`
- `value_numeric`
- `data_level`
- `statistic_type`
- `source_pipeline`
- `hemisphere`

This makes it possible to support both **subject-level** and **group-level** analysis.


In [3]:
def load_optional_csv(path):
    if path is None:
        return None
    path = Path(path)
    if not path.exists():
        return None
    return pd.read_csv(path)


if not STRUCTURAL_FEATURES_CSV_PATH.exists():
    raise FileNotFoundError(f'Structural feature CSV not found: {STRUCTURAL_FEATURES_CSV_PATH}')

structural_df = pd.read_csv(STRUCTURAL_FEATURES_CSV_PATH)
structural_df = normalize_structural_dataframe(structural_df)

normative_df = None
if USE_NORMATIVE_REFERENCE and NORMATIVE_CSV_PATH.exists():
    normative_df = normalize_structural_dataframe(pd.read_csv(NORMATIVE_CSV_PATH))

display(pd.DataFrame([
    {'item': 'structural_row_count', 'value': len(structural_df)},
    {'item': 'structural_column_count', 'value': len(structural_df.columns)},
    {'item': 'normative_loaded', 'value': normative_df is not None},
]))

display(structural_df.head())


,item,value
0,structural_row_count,251
1,structural_column_count,23
2,normative_loaded,False


,subject_id,diagnosis,sex,age,mmse,cdr,roi_name,imaging_metric,value_numeric,data_level,...,unit,source_subject_dir,source_stats_file,metadata_source,metadata_completeness_status,analysis_scope,comparison_role,diagnosis_normalized,roi_name_normalized,imaging_metric_normalized
0,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,lateral ventricle,roi_volume,23869.7,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,lateral ventricle,roi_volume
1,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,inf lat vent,roi_volume,1554.4,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,inf lat vent,roi_volume
2,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,cerebellum white matter,roi_volume,13409.5,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,white matter,roi_volume
3,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,cerebellum cortex,roi_volume,50646.9,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,cerebellum cortex,roi_volume
4,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,thalamus,roi_volume,6109.7,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,thalamus,roi_volume


## Stage 1B. Preview The Structural Feature Layer

### Why this stage matters

In proposal terms, this is the reproducible feature-extraction foundation.

Before building comparisons, it is useful to confirm:

- which diagnoses are represented in the structured table
- which ROIs and imaging metrics are available
- whether the table looks like subject-level data or group summaries


In [4]:
display(Markdown('### Structural feature summary'))
display(summarize_structural_features(structural_df).head(20))

display(Markdown('### Available diagnosis / ROI / metric combinations'))
preview_cols = [c for c in ['subject_id', 'diagnosis', 'diagnosis_normalized', 'roi_name', 'roi_name_normalized', 'imaging_metric', 'imaging_metric_normalized', 'value_numeric', 'source_pipeline'] if c in structural_df.columns]
display(structural_df[preview_cols].head(20))


### Structural feature summary

,diagnosis_normalized,roi_name_normalized,imaging_metric_normalized,n_rows,n_non_missing,mean_value,std_value
0,Dementia,3rd ventricle,roi_volume,1,1,1492.100000,NaN
1,Dementia,4th ventricle,roi_volume,1,1,1999.000000,NaN
2,Dementia,5th ventricle,roi_volume,1,1,0.000000,NaN
3,Dementia,accumbens area,roi_volume,2,2,455.150000,32.597623
4,Dementia,amygdala,roi_volume,3,3,1224.066667,530.036530
5,Dementia,bankssts,cortical_thickness,2,2,2.345000,0.038184
6,Dementia,bankssts,roi_volume,2,2,1784.000000,69.296465
7,Dementia,bankssts,surface_area,2,2,803.500000,44.547727
8,Dementia,brain stem,roi_volume,1,1,17371.600000,NaN
9,Dementia,caudalanteriorcingulate,cortical_thickness,2,2,2.154500,0.136472


### Available diagnosis / ROI / metric combinations

,subject_id,diagnosis,diagnosis_normalized,roi_name,roi_name_normalized,imaging_metric,imaging_metric_normalized,value_numeric,source_pipeline
0,OAS1_0003_MR1,Dementia,Dementia,lateral ventricle,lateral ventricle,roi_volume,roi_volume,23869.7,FreeSurfer 8.2.0 recon-all
1,OAS1_0003_MR1,Dementia,Dementia,inf lat vent,inf lat vent,roi_volume,roi_volume,1554.4,FreeSurfer 8.2.0 recon-all
2,OAS1_0003_MR1,Dementia,Dementia,cerebellum white matter,white matter,roi_volume,roi_volume,13409.5,FreeSurfer 8.2.0 recon-all
3,OAS1_0003_MR1,Dementia,Dementia,cerebellum cortex,cerebellum cortex,roi_volume,roi_volume,50646.9,FreeSurfer 8.2.0 recon-all
4,OAS1_0003_MR1,Dementia,Dementia,thalamus,thalamus,roi_volume,roi_volume,6109.7,FreeSurfer 8.2.0 recon-all
5,OAS1_0003_MR1,Dementia,Dementia,caudate,caudate,roi_volume,roi_volume,3081.9,FreeSurfer 8.2.0 recon-all
6,OAS1_0003_MR1,Dementia,Dementia,putamen,putamen,roi_volume,roi_volume,4202.5,FreeSurfer 8.2.0 recon-all
7,OAS1_0003_MR1,Dementia,Dementia,pallidum,pallidum,roi_volume,roi_volume,1392.8,FreeSurfer 8.2.0 recon-all
8,OAS1_0003_MR1,Dementia,Dementia,3rd ventricle,3rd ventricle,roi_volume,roi_volume,1492.1,FreeSurfer 8.2.0 recon-all
9,OAS1_0003_MR1,Dementia,Dementia,4th ventricle,4th ventricle,roi_volume,roi_volume,1999.0,FreeSurfer 8.2.0 recon-all


## Stage 2. Build Or Load The Literature Evidence Framework

### From scattered findings to structured comparison

This stage corresponds to **Aim 2** in the proposal.

The goal is to convert literature into a reusable framework that links:

- disorder
- region
- MRI feature
- reported pattern
- source
- methodological context

Logic used here:

- If `INPUTS` contains sources, run the extraction pipeline.
- If `INPUTS` is empty, load an existing `literature_database.json`.
- Keep the result structured and traceable, rather than treating literature as free text only.


In [5]:
literature_result = None

if INPUTS:
    literature_result = analyze_sources(
        inputs=INPUTS,
        output_dir=OUTPUT_DIR,
        ai_backend=AI_BACKEND,
        ai_base_url=AI_BASE_URL,
        ai_model=AI_MODEL,
        ai_api_key=AI_API_KEY,
    )
    records = literature_result['records']
else:
    if not LITERATURE_JSON_PATH.exists():
        raise FileNotFoundError(
            f'No INPUTS were provided and no existing literature JSON was found at {LITERATURE_JSON_PATH}'
        )
    records = json.loads(LITERATURE_JSON_PATH.read_text(encoding='utf-8'))

records_df = pd.DataFrame(records)
evidence_df = build_evidence_framework(records)

display(pd.DataFrame([
    {'item': 'literature_record_count', 'value': len(records)},
    {'item': 'used_existing_json', 'value': literature_result is None},
    {'item': 'literature_json_path', 'value': str(LITERATURE_JSON_PATH if literature_result is None else literature_result['json_path'])},
    {'item': 'evidence_framework_rows', 'value': len(evidence_df)},
]))


,item,value
0,literature_record_count,11
1,used_existing_json,True
2,literature_json_path,literature_outputs_workflow_demo/literature_da...
3,evidence_framework_rows,343


## Stage 2A. Preview The Evidence Framework

### Make the evidence structure explicit before comparison

This corresponds to the proposal idea that findings should be **reusable, extendable, and traceable**.

Useful preview views include:

- literature rows by diagnosis
- disease-feature summaries
- framework rows organized by disorder, region, and metric


In [6]:
display(Markdown('### Literature preview columns'))
literature_preview_cols = [
    'title', 'year', 'doi', 'diagnoses', 'rois', 'imaging_metrics',
    'modalities', 'source_type', 'study_design'
]
display(records_df[[c for c in literature_preview_cols if c in records_df.columns]].head(20))

display(Markdown('### Example evidence rows for FTD-spectrum disorders'))
display(query_literature(records, disease='FTD').head(20))

display(Markdown('### Example features reported for Dementia'))
display(literature_features_for_disease(records, 'Dementia').head(20))

display(Markdown('### Reverse search: diseases linked to hippocampus and cortical thickness'))
display(literature_diseases_for_features(records, ['hippocampus', 'cortical_thickness']).head(20))


### Literature preview columns

,title,year,doi,diagnoses,rois,imaging_metrics,modalities,source_type,study_design
0,Classification of primary progressive aphasia ...,2011.0,10.1212/WNL.0b013e31821103e6,"[PPA, svPPA, nfvPPA, lvPPA, FTD]","[anterior temporal cortex, temporal cortex, fr...","[cortical_thickness, roi_volume]",[MRI],pubmed,international consensus clinical diagnostic cr...
1,Sensitivity of revised diagnostic criteria for...,2011.0,10.1093/brain/awr179,"[bvFTD, FTD]","[frontal cortex, anterior temporal cortex, ins...","[cortical_thickness, roi_volume]",[MRI],pubmed,multicentre diagnostic criteria validation
2,Structural Covariance of the Default Network i...,2013.0,10.1523/JNEUROSCI.2261-13.2013,"[Control, MCI, Dementia, AD]","[posterior cingulate cortex, inferior parietal...","[roi_volume, cortical_thickness]",[MRI],pmc,cross-sectional OASIS and longitudinal ADNI st...
3,Longitudinal measures of cholinergic forebrain...,2013.0,10.1016/j.neurobiolaging.2012.10.018,"[Control, MCI, Dementia, AD]","[hippocampus, frontal cortex, temporal cortex]",[roi_volume],[MRI],pmc,longitudinal MRI volumetry study using OASIS-d...
4,MIRIAD--Public release of a multiple time poin...,2013.0,10.1016/j.neuroimage.2012.12.044,"[Control, Dementia, AD]","[hippocampus, temporal cortex, frontal cortex]",[roi_volume],[MRI],pmc,public longitudinal MRI dataset descriptor
5,Automated mapping of hippocampal atrophy in 1-...,2009.0,10.1016/j.neuroimage.2008.10.043,"[Control, MCI, Dementia, AD]",[hippocampus],"[hippocampal_volume, roi_volume]",[MRI],pmc,longitudinal automated hippocampal segmentatio...
6,Automated MRI measures identify individuals wi...,2009.0,10.1093/brain/awp123,"[Control, MCI, Dementia, AD]","[hippocampus, entorhinal cortex, temporal cort...","[cortical_thickness, roi_volume, hippocampal_v...",[MRI],pmc,automated MRI diagnostic classification study
7,Patterns of cortical thinning in the language ...,2009.0,10.1212/WNL.0b013e3181a4124e,"[FTD, PPA, svPPA, nfvPPA]","[temporal cortex, anterior temporal cortex, en...",[cortical_thickness],[MRI],pmc,cross-sectional cortical thickness study using...
8,Peripheral (deep) but not periventricular MRI ...,2016.0,10.1002/brb3.438,"[VaD, Dementia, AD]","[white matter, frontal cortex]","[white_matter_hyperintensity, roi_volume]",[MRI],pmc,clinical MRI white matter hyperintensity compa...
9,"OASIS-3: Longitudinal Neuroimaging, Clinical, ...",2019.0,None,"[Control, MCI, Dementia, AD]","[hippocampus, entorhinal cortex, temporal cort...","[roi_volume, cortical_thickness, hippocampal_v...","[MRI, PET]",dataset,open dataset source for structural MRI and cli...


### Example evidence rows for FTD-spectrum disorders

,paper_id,source_label,title,year,doi,authors,study_design,population,diagnoses,modalities,rois,symptoms,sex_scope,imaging_metrics,findings,limitations,abstract_like_summary,extraction_mode,source_type,source_path_or_url
0,PMID21810890,Gorno-Tempini 2011 PPA variants criteria,Classification of primary progressive aphasia ...,2011,10.1212/WNL.0b013e31821103e6,"[Gorno-Tempini ML, Hillis AE, Weintraub S]",international consensus clinical diagnostic cr...,primary progressive aphasia spectrum,"[PPA, svPPA, nfvPPA, lvPPA, FTD]",[MRI],"[anterior temporal cortex, temporal cortex, fr...",[language disturbance],[],"[cortical_thickness, roi_volume]",[PPA variants are linked to subtype-specific l...,[Consensus criteria support interpretation but...,"Consensus criteria separating semantic, nonflu...",curated_manual_metadata,pubmed,https://pubmed.ncbi.nlm.nih.gov/21810890/
1,PMID21325651,Rascovsky 2011 bvFTD criteria,Sensitivity of revised diagnostic criteria for...,2011,10.1093/brain/awr179,"[Rascovsky K, Hodges JR, Knopman D]",multicentre diagnostic criteria validation,behavioural variant frontotemporal dementia,"[bvFTD, FTD]",[MRI],"[frontal cortex, anterior temporal cortex, ins...","[behavioural change, executive dysfunction, ap...",[],"[cortical_thickness, roi_volume]",[Supportive neuroimaging for bvFTD includes fr...,[Imaging is supportive and must be interpreted...,Validation of revised bvFTD criteria emphasizi...,curated_manual_metadata,pubmed,https://pubmed.ncbi.nlm.nih.gov/21325651/
2,PMC2827264,Rohrer 2009 FTLD language variants,Patterns of cortical thinning in the language ...,2009,10.1212/WNL.0b013e3181a4124e,"[Rohrer JD, Warren JD, Modat M, Ridgway GR, Do...",cross-sectional cortical thickness study using...,"semantic dementia, progressive nonfluent aphas...","[FTD, PPA, svPPA, nfvPPA]",[MRI],"[temporal cortex, anterior temporal cortex, en...",[language disturbance],[],[cortical_thickness],[Semantic dementia and progressive nonfluent a...,[Group-level FTLD language-variant evidence; n...,FreeSurfer cortical thickness analysis of lang...,curated_manual_metadata,pmc,https://pmc.ncbi.nlm.nih.gov/articles/PMC2827264/


### Example features reported for Dementia

,disease,title,roi,metric,modalities,findings
0,Dementia,Structural Covariance of the Default Network i...,posterior cingulate cortex,roi_volume,MRI,Default-network gray matter integrity declines...
1,Dementia,Structural Covariance of the Default Network i...,posterior cingulate cortex,cortical_thickness,MRI,Default-network gray matter integrity declines...
2,Dementia,Structural Covariance of the Default Network i...,inferior parietal cortex,roi_volume,MRI,Default-network gray matter integrity declines...
3,Dementia,Structural Covariance of the Default Network i...,inferior parietal cortex,cortical_thickness,MRI,Default-network gray matter integrity declines...
4,Dementia,Structural Covariance of the Default Network i...,temporal cortex,roi_volume,MRI,Default-network gray matter integrity declines...
5,Dementia,Structural Covariance of the Default Network i...,temporal cortex,cortical_thickness,MRI,Default-network gray matter integrity declines...
6,Dementia,Structural Covariance of the Default Network i...,hippocampus,roi_volume,MRI,Default-network gray matter integrity declines...
7,Dementia,Structural Covariance of the Default Network i...,hippocampus,cortical_thickness,MRI,Default-network gray matter integrity declines...
8,Dementia,Structural Covariance of the Default Network i...,frontal cortex,roi_volume,MRI,Default-network gray matter integrity declines...
9,Dementia,Structural Covariance of the Default Network i...,frontal cortex,cortical_thickness,MRI,Default-network gray matter integrity declines...


### Reverse search: diseases linked to hippocampus and cortical thickness

,title,diagnoses,matched_features,findings
0,Classification of primary progressive aphasia ...,PPA; svPPA; nfvPPA; lvPPA; FTD,"cortical_thickness, hippocampus",PPA variants are linked to subtype-specific la...
1,Sensitivity of revised diagnostic criteria for...,bvFTD; FTD,cortical_thickness,Supportive neuroimaging for bvFTD includes fro...
2,Structural Covariance of the Default Network i...,Control; MCI; Dementia; AD,"cortical_thickness, hippocampus",Default-network gray matter integrity declines...
3,Longitudinal measures of cholinergic forebrain...,Control; MCI; Dementia; AD,hippocampus,Basal forebrain atrophy is greater in very mil...
4,MIRIAD--Public release of a multiple time poin...,Control; Dementia; AD,hippocampus,"MIRIAD provides longitudinal T1 MRI with age, ..."
5,Automated mapping of hippocampal atrophy in 1-...,Control; MCI; Dementia; AD,hippocampus,Hippocampal volume loss rates increase from co...
6,Automated MRI measures identify individuals wi...,Control; MCI; Dementia; AD,"cortical_thickness, hippocampus",Automated MRI measures can distinguish MCI and...
7,Patterns of cortical thinning in the language ...,FTD; PPA; svPPA; nfvPPA,cortical_thickness,Semantic dementia and progressive nonfluent ap...
8,"OASIS-3: Longitudinal Neuroimaging, Clinical, ...",Control; MCI; Dementia; AD,"cortical_thickness, hippocampus",OASIS is the source of the current pilot struc...
9,Alzheimer Disease Neuroimaging Initiative data,AD; MCI; Control; Dementia,"cortical_thickness, hippocampus",ADNI is a planned reference dataset for future...


## Stage 3. Optional Multimodal Extension

### Aim 3 is optional, not a blocker for the core workflow

This stage corresponds to the proposal slide on **complementary information**.

Potential extensions:

- DWI
- perfusion imaging
- cognitive / behavioural data
- blood biomarkers

If these tables are unavailable, the core workflow still remains valid with structural MRI + literature evidence.


In [7]:
dwi_df = load_optional_csv(DWI_CSV_PATH)
perfusion_df = load_optional_csv(PERFUSION_CSV_PATH)
cognitive_df = load_optional_csv(COGNITIVE_CSV_PATH)
biomarker_df = load_optional_csv(BIOMARKER_CSV_PATH)

multimodal_status_df = pd.DataFrame([
    {'table': 'dwi_df', 'loaded': dwi_df is not None},
    {'table': 'perfusion_df', 'loaded': perfusion_df is not None},
    {'table': 'cognitive_df', 'loaded': cognitive_df is not None},
    {'table': 'biomarker_df', 'loaded': biomarker_df is not None},
])

display(multimodal_status_df)


,table,loaded
0,dwi_df,False
1,perfusion_df,False
2,cognitive_df,False
3,biomarker_df,False


## Stage 4. Patient-To-Evidence Comparison

### Match structural features against the framework cautiously

This stage corresponds to the comparison logic in the proposal:

- compare patient or group rows with literature-reported patterns
- optionally compare against normative reference data
- keep methodological caveats visible
- treat the output as **interpretive support**, not diagnosis

The Python helper now supports **combined analysis**:

- structural feature signatures
- literature-linked evidence matches
- finer disease differentiation, including FTD subtypes where possible


In [8]:
display(Markdown('### Example A: Structural rows for Dementia'))
display(query_structural_features(structural_df, diagnosis='Dementia').head(20))

display(Markdown('### Example B: Literature evidence for FTD'))
display(query_literature(records, disease='FTD').head(20))

display(Markdown('### Example C: Match hippocampus ROI features to literature evidence'))
display(compare_features_to_literature(structural_df, evidence_df, roi='hippocampus').head(20))

if USE_NORMATIVE_REFERENCE and normative_df is not None:
    display(Markdown('### Example D: Normative comparison for hippocampus ROI volume'))
    display(compare_to_normative_reference(structural_df, normative_df, roi='hippocampus', metric='roi_volume').head(20))


display(Markdown('### Matching audit columns'))
audit_cols = [c for c in ['subject_id','diagnosis','roi_name','imaging_metric','evidence_match','matching_rule','match_status','manual_review_status','paper_id','source_title','doi'] if c in compare_features_to_literature(structural_df, evidence_df, roi='hippocampus').columns]
display(compare_features_to_literature(structural_df, evidence_df, roi='hippocampus')[audit_cols].head(20))


### Example A: Structural rows for Dementia

,subject_id,diagnosis,sex,age,mmse,cdr,roi_name,imaging_metric,value_numeric,data_level,...,unit,source_subject_dir,source_stats_file,metadata_source,metadata_completeness_status,analysis_scope,comparison_role,diagnosis_normalized,roi_name_normalized,imaging_metric_normalized
0,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,lateral ventricle,roi_volume,23869.7,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,lateral ventricle,roi_volume
1,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,inf lat vent,roi_volume,1554.4,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,inf lat vent,roi_volume
2,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,cerebellum white matter,roi_volume,13409.5,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,white matter,roi_volume
3,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,cerebellum cortex,roi_volume,50646.9,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,cerebellum cortex,roi_volume
4,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,thalamus,roi_volume,6109.7,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,thalamus,roi_volume
5,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,caudate,roi_volume,3081.9,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,caudate,roi_volume
6,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,putamen,roi_volume,4202.5,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,putamen,roi_volume
7,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,pallidum,roi_volume,1392.8,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,pallidum,roi_volume
8,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,3rd ventricle,roi_volume,1492.1,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus available OASIS selected...,incomplete_metadata: sex_unknown_age_mmse_cdr_...,single_case_prototype_no_group_comparison,case_only,Dementia,3rd ventricle,roi_volume
9,OAS1_0003_MR1,Dementia,unknown,NaN,NaN,NaN,4th ventricle,roi_volume,1999.0,individual_subject,...,mm3,/home/jovyan/derivatives/freesurfer/OAS1_0003_MR1,/home/jovyan/derivatives/freesurfer/OAS1_0003_...,FreeSurfer stats plus avail

### Example B: Literature evidence for FTD

,paper_id,source_label,title,year,doi,authors,study_design,population,diagnoses,modalities,rois,symptoms,sex_scope,imaging_metrics,findings,limitations,abstract_like_summary,extraction_mode,source_type,source_path_or_url
0,PMID21810890,Gorno-Tempini 2011 PPA variants criteria,Classification of primary progressive aphasia ...,2011,10.1212/WNL.0b013e31821103e6,"[Gorno-Tempini ML, Hillis AE, Weintraub S]",international consensus clinical diagnostic cr...,primary progressive aphasia spectrum,"[PPA, svPPA, nfvPPA, lvPPA, FTD]",[MRI],"[anterior temporal cortex, temporal cortex, fr...",[language disturbance],[],"[cortical_thickness, roi_volume]",[PPA variants are linked to subtype-specific l...,[Consensus criteria support interpretation but...,"Consensus criteria separating semantic, nonflu...",curated_manual_metadata,pubmed,https://pubmed.ncbi.nlm.nih.gov/21810890/
1,PMID21325651,Rascovsky 2011 bvFTD criteria,Sensitivity of revised diagnostic criteria for...,2011,10.1093/brain/awr179,"[Rascovsky K, Hodges JR, Knopman D]",multicentre diagnostic criteria validation,behavioural variant frontotemporal dementia,"[bvFTD, FTD]",[MRI],"[frontal cortex, anterior temporal cortex, ins...","[behavioural change, executive dysfunction, ap...",[],"[cortical_thickness, roi_volume]",[Supportive neuroimaging for bvFTD includes fr...,[Imaging is supportive and must be interpreted...,Validation of revised bvFTD criteria emphasizi...,curated_manual_metadata,pubmed,https://pubmed.ncbi.nlm.nih.gov/21325651/
2,PMC2827264,Rohrer 2009 FTLD language variants,Patterns of cortical thinning in the language ...,2009,10.1212/WNL.0b013e3181a4124e,"[Rohrer JD, Warren JD, Modat M, Ridgway GR, Do...",cross-sectional cortical thickness study using...,"semantic dementia, progressive nonfluent aphas...","[FTD, PPA, svPPA, nfvPPA]",[MRI],"[temporal cortex, anterior temporal cortex, en...",[language disturbance],[],[cortical_thickness],[Semantic dementia and progressive nonfluent a...,[Group-level FTLD language-variant evidence; n...,FreeSurfer cortical thickness analysis of lang...,curated_manual_metadata,pmc,https://pmc.ncbi.nlm.nih.gov/articles/PMC2827264/


### Example C: Match hippocampus ROI features to literature evidence

,subject_id,diagnosis,roi_name,imaging_metric,value_numeric,evidence_match,matching_rule,match_status,manual_review_status,paper_id,source_title,doi,pattern_summary,limitations
0,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,2897.2,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC3776065,Structural Covariance of the Default Network i...,10.1523/JNEUROSCI.2261-13.2013,Default-network gray matter integrity declines...,Group-level network covariance does not direct...
1,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,2897.2,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC4058576,Longitudinal measures of cholinergic forebrain...,10.1016/j.neurobiolaging.2012.10.018,Basal forebrain atrophy is greater in very mil...,Basal forebrain ROI is not directly produced b...
2,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,2897.2,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC3809512,MIRIAD--Public release of a multiple time poin...,10.1016/j.neuroimage.2012.12.044,"MIRIAD provides longitudinal T1 MRI with age, ...",External dataset source only; not processed in...
3,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,2897.2,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC2733354,Automated mapping of hippocampal atrophy in 1-...,10.1016/j.neuroimage.2008.10.043,Hippocampal volume loss rates increase from co...,Uses specialized automated hippocampal mapping...
4,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,2897.2,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC2714061,Automated MRI measures identify individuals wi...,10.1093/brain/awp123,Automated MRI measures can distinguish MCI and...,"Classification performance depends on cohort, ..."
5,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,2897.2,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,OASIS3,"OASIS-3: Longitudinal Neuroimaging, Clinical, ...",None,OASIS is the source of the current pilot struc...,Current demo uses only a small selected subset.
6,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,2897.2,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,ADNI,Alzheimer Disease Neuroimaging Initiative data,None,ADNI is a planned reference dataset for future...,Not processed in this pilot run.
7,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,2887.4,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC3776065,Structural Covariance of the Default Network i...,10.1523/JNEUROSCI.2261-13.2013,Default-network gray matter integrity declines...,Group-level network covariance does not direct...
8,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,2887.4,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC4058576,Longitudinal measures of cholinergic forebrain...,10.1016/j.neurobiolaging.2012.10.018,Basal forebrain atrophy is greater in very mil...,Basal forebrain ROI is not directly produced b...
9,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,2887.4,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC3809512,MIRIAD--Public release of a multiple time poin...,10.1016/j.neuroimage.2012.12.044,"MIRIAD provides longitudinal T1 MRI with age, ...",External dataset source only; not processed in...


### Matching audit columns

,subject_id,diagnosis,roi_name,imaging_metric,evidence_match,matching_rule,match_status,manual_review_status,paper_id,source_title,doi
0,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC3776065,Structural Covariance of the Default Network i...,10.1523/JNEUROSCI.2261-13.2013
1,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC4058576,Longitudinal measures of cholinergic forebrain...,10.1016/j.neurobiolaging.2012.10.018
2,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC3809512,MIRIAD--Public release of a multiple time poin...,10.1016/j.neuroimage.2012.12.044
3,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC2733354,Automated mapping of hippocampal atrophy in 1-...,10.1016/j.neuroimage.2008.10.043
4,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC2714061,Automated MRI measures identify individuals wi...,10.1093/brain/awp123
5,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,OASIS3,"OASIS-3: Longitudinal Neuroimaging, Clinical, ...",None
6,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,ADNI,Alzheimer Disease Neuroimaging Initiative data,None
7,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC3776065,Structural Covariance of the Default Network i...,10.1523/JNEUROSCI.2261-13.2013
8,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC4058576,Longitudinal measures of cholinergic forebrain...,10.1016/j.neurobiolaging.2012.10.018
9,OAS1_0003_MR1,Dementia,hippocampus,roi_volume,True,exact_normalized_diagnosis_roi_metric,candidate_evidence_match,candidate_requires_manual_review,PMC3809512,MIRIAD--Public release of a multiple time poin...,10.1016/j.neuroimage.2012.12.044


## Stage 4A. Combined Disease Support Analysis

### Combine literature evidence with structural feature patterns

This is the main change in the updated Python workflow.

Disease support is no longer inferred from literature alone. Instead, the workflow combines:

- ROI / metric patterns from the structural feature table
- disease-linked evidence rows from the literature framework
- subtype-aware signatures, especially for the FTD spectrum

This is intended as cautious analytical support rather than definitive classification.


In [9]:
display(Markdown('### Conservative single-case candidate support ranking'))
diagnosis_support_df = summarize_diagnosis_support(structural_df, records, subject_id='OAS1_0003_MR1')
display(diagnosis_support_df.head(20))


### Conservative single-case candidate support ranking

,diagnosis,measured_structural_rows,structural_signature_score,signature_hits,literature_match_rows,combined_support_score,ranking_interpretation,scope_note
0,Dementia,251,0,not_used_in_single_case_prototype,63,63,candidate support only; not diagnostic,single-case prototype; no group comparison or ...
1,AD,251,0,not_used_in_single_case_prototype,0,0,no matched evidence in current single-case fea...,single-case prototype; no group comparison or ...
2,Control,251,0,not_used_in_single_case_prototype,0,0,no matched evidence in current single-case fea...,single-case prototype; no group comparison or ...
3,FTD,251,0,not_used_in_single_case_prototype,0,0,no matched evidence in current single-case fea...,single-case prototype; no group comparison or ...
4,MCI,251,0,not_used_in_single_case_prototype,0,0,no matched evidence in current single-case fea...,single-case prototype; no group comparison or ...
5,PPA,251,0,not_used_in_single_case_prototype,0,0,no matched evidence in current single-case fea...,single-case prototype; no group comparison or ...
6,VaD,251,0,not_used_in_single_case_prototype,0,0,no matched evidence in current single-case fea...,single-case prototype; no group comparison or ...
7,bvFTD,251,0,not_used_in_single_case_prototype,0,0,no matched evidence in current single-case fea...,single-case prototype; no group comparison or ...
8,lvPPA,251,0,not_used_in_single_case_prototype,0,0,no matched evidence in current single-case fea...,single-case prototype; no group comparison or ...
9,nfvPPA,251,0,not_used_in_single_case_prototype,0,0,no matched evidence in current single-case fea...,single-case prototype; no group comparison or ...


## Stage 5. Structured Interpretive Report

### Output a cautious, traceable summary

This stage corresponds to the **structured report** described in the proposal slides.

The report should:

- restate what structural features were examined
- list evidence-linked matches
- summarize disease support ranking
- flag missing information
- keep uncertainty and methodological caveats explicit


In [10]:
report_text = build_interpretive_report(
    structural_df=structural_df,
    records=records,
    subject_id='OAS1_0003_MR1',
    diagnosis='Dementia',
    roi='hippocampus',
)
display(Markdown(report_text))


# Structured Interpretive Report

## Scope
- subject_id: OAS1_0003_MR1
- diagnosis filter: Dementia
- roi filter: hippocampus
- imaging_metric filter: None

## Structural MRI Summary
- matched structural rows: 9
- source pipelines: FreeSurfer 8.2.0 recon-all

## Candidate Literature-Linked Evidence (manual review required)
- evidence-linked rows: 45
- example supporting sources:
  - Structural Covariance of the Default Network in Healthy and Pathological Aging
  - Longitudinal measures of cholinergic forebrain atrophy in the transition from healthy aging to Alzheimer disease
  - MIRIAD--Public release of a multiple time point Alzheimer MR imaging dataset
  - Automated mapping of hippocampal atrophy in 1-year repeat MRI data from 490 subjects with Alzheimer disease, mild cognitive impairment, and elderly controls
  - Automated MRI measures identify individuals with mild cognitive impairment and Alzheimer disease

## Diagnosis Support Ranking
- Dementia: combined=43 (structural=0, literature=43)
- AD: combined=0 (structural=0, literature=0)
- Control: combined=0 (structural=0, literature=0)
- FTD: combined=0 (structural=0, literature=0)
- MCI: combined=0 (structural=0, literature=0)

## Interpretation Notes
- This workflow provides evidence-informed interpretive support, not automated diagnosis.
- Disease support is derived from both structural-feature signatures and literature-linked matches.
- Pipeline choice, preprocessing variability, cohort context, and incomplete literature coverage should remain visible in interpretation.

## Suggested Presentation Order

A proposal-aligned walkthrough is:

1. Show the configured structural MRI, literature, and optional multimodal inputs.
2. Show the structural feature table and explain the role of FreeSurfer / FastSurfer outputs.
3. Build or load the literature evidence framework.
4. Preview the evidence schema by disorder, region, and metric.
5. Run one comparison example between structural rows and literature evidence.
6. Show the combined disease-support analysis.
7. If available, show a normative comparison.
8. End with the structured interpretive report and caveats.

This sequence now matches the logic presented in the proposal and thesis slides.
